# NB3 - GeoCalib (camera calibration from a single image)
Input : ROOT/images/*
Output: ROOT/out/camera/<stem>.json  -> focal length, principal point, gravity vector (in camera frame)
The gravity vector defines the ground plane used by the geometry distance method in NB4.
Runtime: GPU (T4)

In [ ]:
!pip -q install git+https://github.com/cvg/GeoCalib.git
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json
import numpy as np, torch
from geocalib import GeoCalib

# ---- CHOOSE WHERE TO WORK (same two lines as in the Depth Pro notebook) ----
TARGET    = 'demo'   # 'accuracy' = tape-measured photos | 'demo' = images for the final pipeline
OVERWRITE = False        # True = recompute even if an output already exists
# -----------------------------------------------------------------------------
BASE    = f'/content/drive/MyDrive/cv5570_risk/{TARGET}'
IMG_DIR = f'{BASE}/images'
OUT_C   = f'{BASE}/camera'
for d in (IMG_DIR, OUT_C): os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = GeoCalib().to(device)
print('images folder:', IMG_DIR)

In [ ]:
paths = sorted(p for p in glob.glob(f'{IMG_DIR}/*') if p.lower().endswith(('.jpg', '.jpeg', '.png')))
print(len(paths), 'images in', IMG_DIR)
assert paths, 'No images found - put them in the images folder first'

for p in paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    jp = f'{OUT_C}/{stem}.json'
    try:
        img = model.load_image(p).to(device)           # tensor 3xHxW at original resolution
    except Exception as ex:
        print(f'{stem}: cannot read this file ({type(ex).__name__}) - skipped'); continue
    H, W = int(img.shape[-2]), int(img.shape[-1])

    if os.path.exists(jp) and not OVERWRITE:
        old = json.load(open(jp))
        if (old['width'], old['height']) == (W, H):
            print(f'{stem}: already done (skipped)'); continue
        print(f'{stem}: saved calibration has a different image size -> recomputing')

    with torch.no_grad():
        res = model.calibrate(img)
    cam, grav = res['camera'], res['gravity']
    f = cam.f[0].cpu().numpy(); c = cam.c[0].cpu().numpy()
    g = grav.vec3d[0].cpu().numpy().astype(float)      # unit vector, camera frame (x right, y down, z forward). It points UP:
    g = g / np.linalg.norm(g)                          #   an upright photo gives g ~ (0, -1, 0). The next notebooks negate it.
    roll, pitch = [float(np.degrees(v)) for v in grav.rp[0].cpu().numpy()]
    out = dict(stem=stem, width=W, height=H, fx=float(f[0]), fy=float(f[1]), cx=float(c[0]), cy=float(c[1]),
               g=g.tolist(), g_convention='up vector, camera frame x right / y down / z forward',
               roll_deg=roll, pitch_deg=pitch,
               near_topdown=bool(-g[2] > 0.85))        # camera looking (almost) straight DOWN -> geometry unreliable
    json.dump(out, open(jp, 'w'))
    print(f'{stem}: saved | f={f[0]:.0f}px roll={roll:.1f} pitch={pitch:.1f} g={np.round(g, 2)} topdown={out["near_topdown"]}')

## Sanity checks (do these once)
* For a normal upright photo, g should be roughly (0, +0.9..1.0, small): y component large and positive.
  If it is negative, the sign convention differs in your installed GeoCalib version - tell me and I will adjust NB4.
* |roll| should be small (a few degrees) for handheld shots; pitch is how far the camera is tilted down.
* For a FIXED camera (video later) run this once on one frame and reuse the json.

In [ ]:
# ===== CHECK: do images / depth maps / camera files line up?  (add at the end of either notebook; run after BOTH have run) =====
import os, glob, json
import numpy as np, cv2, pandas as pd
TARGET = 'demo'                      # same value as above
BASE = f'/content/drive/MyDrive/cv5570_risk/{TARGET}'

imgs = sorted(p for p in glob.glob(f'{BASE}/images/*') if p.lower().endswith(('.jpg', '.jpeg', '.png')))
rows = []
for p in imgs:
    stem = os.path.splitext(os.path.basename(p))[0]
    im = cv2.imread(p)
    if im is None:
        rows.append(dict(image=os.path.basename(p), size='UNREADABLE', depth='-', camera='-')); continue
    H, W = im.shape[:2]
    r = dict(image=os.path.basename(p), size=f'{W}x{H}', depth='MISSING', camera='MISSING')
    dz, cj = f'{BASE}/depth/{stem}.npz', f'{BASE}/camera/{stem}.json'
    if os.path.exists(dz):
        z = np.load(dz); r['depth'] = 'ok' if z['depth'].shape == (H, W) else 'SIZE MISMATCH'; r['f_depthpro'] = round(float(z['f_px']))
    if os.path.exists(cj):
        c = json.load(open(cj)); r['camera'] = 'ok' if (c['width'], c['height']) == (W, H) else 'SIZE MISMATCH'
        r['f_geocalib'] = round(c['fx']); r['pitch_deg'] = round(c['pitch_deg'], 1)
        r['g_y (expect ~ -1)'] = round(c['g'][1], 2)
    if 'f_depthpro' in r and 'f_geocalib' in r: r['f_ratio'] = round(r['f_depthpro'] / r['f_geocalib'], 2)
    rows.append(r)
df = pd.DataFrame(rows); display(df)

stems = {os.path.splitext(os.path.basename(p))[0] for p in imgs}
orphans = [os.path.basename(x) for d, e in (('depth', 'npz'), ('camera', 'json')) for x in glob.glob(f'{BASE}/{d}/*.{e}')
           if os.path.splitext(os.path.basename(x))[0] not in stems]
if orphans: print('outputs with no matching image (old runs / different names):', orphans)
print('ALL GOOD - ready for the next notebooks' if not orphans and (df[['depth', 'camera']] == 'ok').all().all()
      else 'FIX the MISSING / MISMATCH / UNREADABLE rows above before moving on')

In [ ]:
import pandas as pd
import glob
import json
import os

# Load and display all calibration JSON outputs
json_paths = sorted(glob.glob(f'{OUT_C}/*.json'))
records = []

for jp in json_paths:
    with open(jp, 'r') as f:
        records.append(json.load(f))

if records:
    df_cam = pd.DataFrame(records)
    display(df_cam)
else:
    print('No camera calibration JSON files found.')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Visualize the images with their gravity vectors mapped onto them
max_images_to_show = 5
show_paths = paths[:max_images_to_show]

for p in show_paths:
    stem = os.path.splitext(os.path.basename(p))[0]
    json_path = f'{OUT_C}/{stem}.json'

    if not os.path.exists(json_path):
        continue

    with open(json_path, 'r') as f:
        data = json.load(f)

    img = Image.open(p)
    w, h = img.size

    plt.figure(figsize=(8, 6))
    plt.imshow(img)

    # Plot principal point
    plt.plot(data['cx'], data['cy'], 'ro', label='Principal Point')

    # Draw a 2D projection of the gravity vector (g_x, g_y) starting from the center
    g = data['g']
    start_x, start_y = data['cx'], data['cy']
    # Scale vector for visual presentation
    scale = min(w, h) * 0.2
    end_x = start_x + g[0] * scale
    end_y = start_y + g[1] * scale

    plt.arrow(start_x, start_y, end_x - start_x, end_y - start_y,
              color='yellow', head_width=min(w, h)*0.02, label='Gravity (down)')

    plt.title(f'{stem} (Roll: {data["roll_deg"]:.1f}°, Pitch: {data["pitch_deg"]:.1f}°)')
    plt.legend()
    plt.axis('on')
    plt.show()